# Steps 2-4 — Clean GTFS Transit Data + Geo-Match to Zones
**Urban Pulse: Smart City Mobility Intelligence Platform**

Upload these 6 files to Colab first: `routes.txt`, `stops.txt`, `trips.txt`, 
`stop_times.txt`, `agency.txt`, `calendar.txt`

In [13]:
# STEPS 2-4: Clean GTFS transit files + geo-match stops to Zones
# Urban Pulse - Smart City Mobility Intelligence Platform

In [14]:
import pandas as pd
import numpy as np

routes = pd.read_csv(r"c:\infosys internship task\archive (13)\routes.txt")
stops = pd.read_csv(r"c:\infosys internship task\archive (13)\stops.txt")
trips = pd.read_csv(r"c:\infosys internship task\archive (13)\trips.txt")
stop_times = pd.read_csv(r"c:\infosys internship task\archive (13)\stop_times.txt")

In [15]:
# STEP 2: Clean each file

In [16]:
# Strip whitespace from every text column (fixes '1A ' vs '1A' type bugs)
for df in [routes, stops, trips, stop_times]:
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()

# Drop fully empty/unused columns that add noise
routes = routes.drop(columns=[c for c in
    ["route_desc", "route_url", "dummy", "Frequency/Timings source",
     "Frequency timings destination", "Comments"] if c in routes.columns])

stops = stops.drop(columns=[c for c in
    ["stop_desc", "zone_id", "searchlink", "validation_url", "moderated?(y)"]
    if c in stops.columns])

# Flag which routes actually have trip-level schedule data
routes_with_schedule = set(trips["route_id"].unique())
routes["has_schedule"] = routes["route_id"].isin(routes_with_schedule)

print(f"Routes total: {len(routes)}")
print(f"Routes WITH schedule data: {routes['has_schedule'].sum()} "
      f"({routes['has_schedule'].mean()*100:.1f}%)")
print(f"Routes WITHOUT schedule data: {(~routes['has_schedule']).sum()}")

# Coerce stop coordinates to numeric, drop any that fail
stops["stop_lat"] = pd.to_numeric(stops["stop_lat"], errors="coerce")
stops["stop_lon"] = pd.to_numeric(stops["stop_lon"], errors="coerce")
before = len(stops)
stops = stops.dropna(subset=["stop_lat", "stop_lon"])
print(f"\nStops dropped for bad coordinates: {before - len(stops)}")
print(f"Stops remaining: {len(stops)}")

Routes total: 2199
Routes WITH schedule data: 136 (6.2%)
Routes WITHOUT schedule data: 2063

Stops dropped for bad coordinates: 0
Stops remaining: 716


In [17]:
# STEP 3: Geo-match each stop to its nearest Zone

In [18]:
# Approximate zone centroids (rough estimates from general geography —
# verify/refine later with an actual BBMP zone shapefile if precision matters)
zone_centroids = {
    "Bangalore East":                (12.9784, 77.6408),
    "Bommanahalli":                  (12.9077, 77.6365),
    "Bangalore South":               (12.9250, 77.5938),
    "Mahadevapura":                  (12.9698, 77.7500),
    "Byatarayanapura":               (13.0450, 77.5950),
    "Dasarahalli":                   (13.0350, 77.5150),
    "Bangalore West":                (12.9750, 77.5350),
    "Bangalore North Taluk (BIAAPA)":(13.1986, 77.7066),
    "Rajarajeshwari Nagara":         (12.9280, 77.5100),
    "Devanahalli Taluk (BIAAPA)":    (13.2500, 77.7150),
    "Doddaballapura Taluk (BIAAPA)": (13.2940, 77.5350),
}

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def nearest_zone(lat, lon):
    dists = {z: haversine_km(lat, lon, c[0], c[1]) for z, c in zone_centroids.items()}
    best_zone = min(dists, key=dists.get)
    return pd.Series([best_zone, round(dists[best_zone], 2)])

stops[["Zone", "distance_km_to_centroid"]] = stops.apply(
    lambda r: nearest_zone(r["stop_lat"], r["stop_lon"]), axis=1)

print("\nStops per Zone:")
print(stops["Zone"].value_counts())

# Flag stops that are suspiciously far from ANY centroid (>15km = likely
# outside Bangalore metro or a bad coordinate) so you can sanity-check them
far_stops = stops[stops["distance_km_to_centroid"] > 15]
print(f"\nStops >15km from nearest zone centroid (review these manually): "
      f"{len(far_stops)}")


Stops per Zone:
Zone
Bangalore South                   133
Bangalore East                    109
Bangalore West                     97
Byatarayanapura                    94
Bommanahalli                       77
Rajarajeshwari Nagara              73
Mahadevapura                       55
Dasarahalli                        54
Bangalore North Taluk (BIAAPA)     19
Devanahalli Taluk (BIAAPA)          4
Doddaballapura Taluk (BIAAPA)       1
Name: count, dtype: int64

Stops >15km from nearest zone centroid (review these manually): 17


In [19]:
# STEP 4: Roll the Zone assignment up to routes and save

In [20]:
# A route's Zone = the most common Zone among its stops
route_stop_zone = stop_times.merge(
    stops[["stop_id", "Zone"]], on="stop_id", how="left"
).merge(
    trips[["trip_id", "route_id"]], on="trip_id", how="left"
)

route_zone = (route_stop_zone.dropna(subset=["Zone"])
              .groupby("route_id")["Zone"]
              .agg(lambda x: x.value_counts().idxmax())
              .reset_index())

print(f"\nRoutes with a resolved Zone (from schedule data): {len(route_zone)}")

# Save cleaned outputs — reuse these in the next notebook
routes.to_csv("routes_clean.csv", index=False)
stops.to_csv("stops_with_zone.csv", index=False)
trips.to_csv("trips_clean.csv", index=False)
stop_times.to_csv("stop_times_clean.csv", index=False)
route_zone.to_csv("route_zone_mapping.csv", index=False)

print("\nSaved: routes_clean.csv, stops_with_zone.csv, trips_clean.csv, "
      "stop_times_clean.csv, route_zone_mapping.csv")


Routes with a resolved Zone (from schedule data): 70

Saved: routes_clean.csv, stops_with_zone.csv, trips_clean.csv, stop_times_clean.csv, route_zone_mapping.csv


In [1]:
import pandas as pd
import numpy as np
import os

os.chdir(r"c:\infosys internship task\archive (13)")

routes = pd.read_csv("routes.txt")
stops = pd.read_csv("stops.txt")
trips = pd.read_csv("trips.txt")
stop_times = pd.read_csv("stop_times.txt")

# --- Clean ---
for df in [routes, stops, trips, stop_times]:
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()

routes_with_schedule = set(trips["route_id"].unique())
routes["has_schedule"] = routes["route_id"].isin(routes_with_schedule)

stops["stop_lat"] = pd.to_numeric(stops["stop_lat"], errors="coerce")
stops["stop_lon"] = pd.to_numeric(stops["stop_lon"], errors="coerce")
stops = stops.dropna(subset=["stop_lat", "stop_lon"])

# --- Geo-match stops to Zone ---
zone_centroids = {
    "Bangalore East": (12.9784, 77.6408), "Bommanahalli": (12.9077, 77.6365),
    "Bangalore South": (12.9250, 77.5938), "Mahadevapura": (12.9698, 77.7500),
    "Byatarayanapura": (13.0450, 77.5950), "Dasarahalli": (13.0350, 77.5150),
    "Bangalore West": (12.9750, 77.5350),
    "Bangalore North Taluk (BIAAPA)": (13.1986, 77.7066),
    "Rajarajeshwari Nagara": (12.9280, 77.5100),
    "Devanahalli Taluk (BIAAPA)": (13.2500, 77.7150),
    "Doddaballapura Taluk (BIAAPA)": (13.2940, 77.5350),
}

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def nearest_zone(lat, lon):
    dists = {z: haversine_km(lat, lon, c[0], c[1]) for z, c in zone_centroids.items()}
    best = min(dists, key=dists.get)
    return pd.Series([best, round(dists[best], 2)])

stops[["Zone", "distance_km_to_centroid"]] = stops.apply(
    lambda r: nearest_zone(r["stop_lat"], r["stop_lon"]), axis=1)

# --- Roll Zone up to routes ---
route_stop_zone = stop_times.merge(
    stops[["stop_id", "Zone"]], on="stop_id", how="left"
).merge(trips[["trip_id", "route_id"]], on="trip_id", how="left")

route_zone = (route_stop_zone.dropna(subset=["Zone"])
              .groupby("route_id")["Zone"]
              .agg(lambda x: x.value_counts().idxmax())
              .reset_index())

# --- Save — guaranteed same folder since we os.chdir'd above ---
routes.to_csv("routes_clean.csv", index=False)
stops.to_csv("stops_with_zone.csv", index=False)
trips.to_csv("trips_clean.csv", index=False)
stop_times.to_csv("stop_times_clean.csv", index=False)
route_zone.to_csv("route_zone_mapping.csv", index=False)

print("Saved to:", os.getcwd())
print(os.listdir())

Saved to: c:\infosys internship task\archive (13)
['agency.txt', 'calendar.txt', 'routes.txt', 'routes_clean.csv', 'route_zone_mapping.csv', 'stops.txt', 'stops_with_zone.csv', 'stop_times.txt', 'stop_times_clean.csv', 'trips.txt', 'trips_clean.csv']
